# Response Analysis Notebook

This notebook analyzes responses from the code generation agent to improve performance and quality.

In [ ]:
# Import necessary libraries
import sys
import os
import json
import re
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

sys.path.append('..')

from src.utils.token_counter import global_token_counter
from src.utils.logger import get_logger

## Response Quality Metrics

In [ ]:
def analyze_code_response(response_text):
    """Analyze the quality of a code generation response."""
    metrics = {
        'has_code_block': bool(re.search(r'```\w*\n[\s\S]*?```', response_text)),
        'has_comments': bool(re.search(r'#.*|//.*|/\*[\s\S]*?\*/', response_text)),
        'has_error_handling': bool(re.search(r'try|except|catch|throw|error', response_text, re.IGNORECASE)),
        'has_type_hints': bool(re.search(r':\s*\w+|->\s*\w+', response_text)),
        'word_count': len(response_text.split()),
        'line_count': len(response_text.split('\n')),
        'code_to_text_ratio': len(re.findall(r'```[\s\S]*?```', response_text)) / max(1, len(response_text.split()))
    }
    return metrics

# Example analysis
sample_response = """
Here's a Python function to calculate factorial:

```python
def factorial(n: int) -> int:
    """Calculate factorial of a number."""
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers")
    if n == 0 or n == 1:
        return 1
    return n * factorial(n - 1)
```

This function includes error handling for negative inputs and uses recursion.
"""

metrics = analyze_code_response(sample_response)
print("Response Quality Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value}")

## Token Usage Analysis

In [ ]:
# Analyze token usage patterns
usage_summary = global_token_counter.get_usage_summary()

print("Token Usage Summary:")
print(f"Total requests: {usage_summary['request_count']}")
print(f"Total sessions: {usage_summary['session_count']}")

if usage_summary['total_usage'].total_tokens > 0:
    total = usage_summary['total_usage']
    print(f"\nTotal tokens: {total.total_tokens}")
    print(f"Input tokens: {total.input_tokens}")
    print(f"Output tokens: {total.output_tokens}")
    if total.cost_estimate:
        print(f"Estimated cost: ${total.cost_estimate:.4f}")
else:
    print("No token usage data available yet.")

## Response Time Analysis

In [ ]:
# Simulate response time data for analysis
import numpy as np

# Generate sample response times (in seconds)
response_times = np.random.normal(3.5, 1.2, 100)  # Mean 3.5s, std 1.2s
response_times = np.clip(response_times, 0.5, 15)  # Clip to reasonable range

# Create DataFrame for analysis
df = pd.DataFrame({
    'response_time': response_times,
    'request_type': np.random.choice(['simple', 'complex', 'very_complex'], 100, p=[0.5, 0.3, 0.2])
})

# Statistical analysis
print("Response Time Statistics:")
print(df['response_time'].describe())

print("\nBy Request Type:")
print(df.groupby('request_type')['response_time'].agg(['mean', 'std', 'count']))

## Visualization

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Response time distribution
axes[0, 0].hist(response_times, bins=20, alpha=0.7, color='skyblue')
axes[0, 0].set_title('Response Time Distribution')
axes[0, 0].set_xlabel('Response Time (seconds)')
axes[0, 0].set_ylabel('Frequency')

# Response time by request type
df.boxplot(column='response_time', by='request_type', ax=axes[0, 1])
axes[0, 1].set_title('Response Time by Request Type')
axes[0, 1].set_xlabel('Request Type')
axes[0, 1].set_ylabel('Response Time (seconds)')

# Token usage by model (simulated)
models = ['gemini-2.5-flash', 'gpt-4', 'claude-3']
token_usage = [1500, 1200, 1800]
axes[1, 0].bar(models, token_usage, color=['green', 'blue', 'orange'])
axes[1, 0].set_title('Token Usage by Model')
axes[1, 0].set_ylabel('Average Tokens')
axes[1, 0].tick_params(axis='x', rotation=45)

# Quality metrics (simulated)
quality_metrics = ['Code Block', 'Comments', 'Error Handling', 'Type Hints']
quality_scores = [0.95, 0.78, 0.65, 0.82]
axes[1, 1].bar(quality_metrics, quality_scores, color='lightcoral')
axes[1, 1].set_title('Code Quality Metrics')
axes[1, 1].set_ylabel('Presence Rate')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Error Analysis

In [ ]:
# Analyze common error patterns
error_patterns = {
    'syntax_errors': ['SyntaxError', 'IndentationError', 'invalid syntax'],
    'logic_errors': ['infinite loop', 'wrong output', 'incorrect logic'],
    'runtime_errors': ['NameError', 'TypeError', 'ValueError'],
    'incomplete_code': ['TODO', 'placeholder', 'not implemented']
}

def categorize_errors(response_text):
    """Categorize errors in response text."""
    errors = {}
    for category, patterns in error_patterns.items():
        count = sum(1 for pattern in patterns if pattern.lower() in response_text.lower())
        if count > 0:
            errors[category] = count
    return errors

# Example error analysis
sample_responses = [
    "def func(): return x  # NameError: x not defined",
    "for i in range(10) print(i)  # SyntaxError: missing colon",
    "# TODO: implement this function",
    "def factorial(n): return n * factorial(n-1)  # infinite loop for n=0"
]

print("Error Analysis:")
for i, response in enumerate(sample_responses, 1):
    errors = categorize_errors(response)
    print(f"Response {i}: {errors if errors else 'No errors detected'}")

## Improvement Recommendations

In [ ]:
def generate_recommendations(metrics, errors):
    """Generate improvement recommendations based on analysis."""
    recommendations = []
    
    if not metrics.get('has_error_handling', False):
        recommendations.append("Add more error handling examples to training data")
    
    if not metrics.get('has_comments', False):
        recommendations.append("Emphasize code documentation in prompts")
    
    if metrics.get('code_to_text_ratio', 0) < 0.3:
        recommendations.append("Increase code-to-explanation ratio")
    
    if 'syntax_errors' in errors:
        recommendations.append("Improve syntax validation in post-processing")
    
    if 'incomplete_code' in errors:
        recommendations.append("Add completion validation to ensure full implementations")
    
    return recommendations

# Generate recommendations based on sample data
sample_metrics = {'has_error_handling': False, 'has_comments': True, 'code_to_text_ratio': 0.2}
sample_errors = {'syntax_errors': 1, 'incomplete_code': 1}

recommendations = generate_recommendations(sample_metrics, sample_errors)

print("Improvement Recommendations:")
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")